# Taller LangChain + LangSmith: puntos 6 al 10 con Gemma 3 local

Adaptación del notebook original. **Ejecutar en VS Code** con el kernel de `.venv` y Ollama activo.

Usamos el modelo **`gemma3:1b`** ya instalado. Para evitar problemas de *native tool calling* con esa versión, las herramientas se orquestan mediante grafos de LangGraph; Gemma escribe las respuestas. La distinción frente a `create_agent` autónomo se explica en `README.md`.

## Preparación (una vez por entorno)

Primero abre la carpeta completa en VS Code, crea `.venv` e instala `requirements.txt` como se describe en `README.md`. Edita `.env` con `OLLAMA_MODEL=gemma3:1b` y, si deseas registrar trazas, con la clave y el agente de LangSmith. Reinicia el kernel si modificas `.env`.

La siguiente celda sirve para verificar la conexión directa a Ollama.

In [ ]:
from configuracion import crear_modelo, MODELO_NOMBRE

modelo = crear_modelo()
print("Modelo local:", MODELO_NOMBRE)
print(modelo.invoke("Di hola en una frase en español.").content)

## 6. Herramientas y agentes: clima y multiplicación

En el original se definen dos herramientas `@tool` y un agente `create_agent`. Aquí conservamos las herramientas, pero las ejecutamos en un flujo de LangGraph compatible con el Gemma estándar. El clima es **simulado** por defecto (24 °C soleado), como en el original; puedes poner `CLIMA_REAL=true` en `.env` para usar Open-Meteo.

Prueba **las dos operaciones en una misma solicitud**:

In [ ]:
from herramientas import clima, multiplicar
from agentes import consultar_herramientas

print("Herramienta multiplicar directamente:", multiplicar.invoke({"a": 23.5, "b": 4}))
resultado_6 = consultar_herramientas("¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?")
print("\nHerramientas usadas:")
for dato in resultado_6["resultados_herramientas"]:
    print("-", dato)
print("\nRespuesta final:", resultado_6["respuesta"])

**Revisa LangSmith:** abre la traza `Punto_6_agente_clima_multiplicar`, entra en `usar_herramientas` y observa las herramientas `clima` y `multiplicar`; después mira la llamada al modelo local. Las trazas solo se envían si activaste `LANGSMITH_TRACING`.

## 7. RAG básico con BM25

Se conservan las 4 políticas ficticias del tutorial, ahora almacenadas en `datos/politicas_ejemplo.txt`. El recuperador no requiere embeddings ni API externa; el modelo genera respuesta **solamente con el contexto recuperado**. Se incluyen los fragmentos para poder comprobarlos.

In [ ]:
from agentes import consultar_rag

resultado_7 = consultar_rag("¿Cuántos días de remoto puedo hacer?")
print("CONTEXTO RECUPERADO:\n", resultado_7["contexto"])
print("\nRESPUESTA:\n", resultado_7["respuesta"])

In [ ]:
# Prueba de pregunta fuera de los documentos: debe reconocer que no dispone de información.
resultado_sin_evidencia = consultar_rag("¿Cuál es el presupuesto del proyecto Moka?")
print(resultado_sin_evidencia["respuesta"])

## 8. Traductor + `@traceable`

El notebook original usa `@traceable` en el punto 8; el **traductor es un ejercicio adicional solicitado** y usa esa misma funcionalidad. La función `traducir` está decorada para trazar entrada, salida y pasos del pipeline.

In [ ]:
from agentes import traducir

print(traducir(
    texto="El presupuesto del proyecto está listo para revisión.",
    origen="español",
    destino="inglés",
))
print(traducir(
    texto="The construction schedule needs an update.",
    origen="inglés",
    destino="español",
))

## 9. Evaluación con datasets en LangSmith

El programa `evaluar.py` crea un dataset de 3 preguntas sobre políticas y usa un evaluador simple de coincidencia de respuesta. **Necesita API key de LangSmith con permisos para datasets y experimentos.** La siguiente celda solo ejecuta la evaluación si hay credenciales configuradas.

In [ ]:
import os
from configuracion import RAIZ

if os.getenv("LANGSMITH_API_KEY"):
    from evaluar import main as evaluar_en_langsmith
    evaluar_en_langsmith()
else:
    print("Configura LANGSMITH_API_KEY en .env y reinicia el kernel para evaluar.")

## 10. Memoria conversacional con `InMemorySaver`

Con el mismo `thread_id`, el historial permanece entre dos invocaciones dentro de una sesión de Python. No es memoria persistente en disco.

In [ ]:
from memoria import demostracion_memoria

print(demostracion_memoria())

### Alternativa opcional: `create_agent` nativo

Si el profesor exige exactamente la llamada `create_agent` del punto 6, revisa `agente_nativo_opcional.py` y el apartado 8 de `README.md`. **Necesita descargar** `lukaspetrik/gemma3-tools:1b`; no es el mismo archivo de modelo `gemma3:1b` ya instalado. No se ha validado su comportamiento en tu máquina.